## Export Variant Video Library

- pull these functions into the main library later

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
from dask.distributed import wait

import scipy.stats
from sklearn.linear_model import LinearRegression

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [ ]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

In [ ]:
import h5py
import cv2
import skimage as sk
from skimage.color import gray2rgb
import copy


def lookups_to_output_arr(headpath,x):
    for i in range(len(x)):
        with h5py.File(headpath + "/kymograph/kymograph_" + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
            if i == 0:
                output_arr_dict = {channel:[] for channel in list(infile.keys())}
            for channel in list(infile.keys()):
                if x.iloc[i]["lane orientation"] == "top":
                    output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
                else:
                    output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
    for channel in list(output_arr_dict.keys()):
        output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
    return output_arr_dict

def export_avi(output_arr,outpath,min_intensity = 0,max_intensity = 10000, fps=10):
    output_arr_scaled = sk.exposure.rescale_intensity(output_arr,in_range=(min_intensity,max_intensity))
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(outpath,fourcc, fps, (output_arr_scaled.shape[2],output_arr_scaled.shape[1]))
    for t in range(output_arr_scaled.shape[0]):
        out.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))
    out.release()
    
def lookup_to_avi(headpath_list,outpath,channel_params_dict,x):
    tr.writedir(outpath, overwrite=False)
    
    output_arr_dict_list = []
    for i,headpath in enumerate(headpath_list):
        filtered_x = x[x["Experiment #"]==i]
        if len(filtered_x)>0:
            output_arr_dict = lookups_to_output_arr(headpath,filtered_x)
            output_arr_dict_list.append(output_arr_dict)
        
    merged_output_arr_dict = {}
    for channel in list(output_arr_dict.keys()):
        merged_output_arr_dict[channel] = np.concatenate([output_arr_dict[channel] for output_arr_dict in output_arr_dict_list], axis=2)
    del output_arr_dict_list
        
    for channel in list(merged_output_arr_dict.keys()):
        selected_channel_params = channel_params_dict[channel]
        min_intensity,max_intensity,fps = selected_channel_params
        tr.writedir(outpath + "/" + channel, overwrite=False)
        export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".avi",\
                   min_intensity = min_intensity, max_intensity = max_intensity, fps=fps)

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=200,
    n_workers_min=50,
    memory="4GB",
    working_directory="/home/lag36/scratch/lag36/dask",
)
dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

In [ ]:
output_phenotype_kmyo_df_path_list = ["/home/de64/scratch/de64/sync_folder/2022-08-10_lDE23_LowSampling_Merged_Analysis/2022-08-10_lDE23_Final_Barcodes_df_Run_1/",\
                         "/home/de64/scratch/de64/sync_folder/2022-08-10_lDE23_LowSampling_Merged_Analysis/2022-08-10_lDE23_Final_Barcodes_df_Run_2/"]
output_phenotype_kmyo_df_list = [dd.read_parquet(item,engine="pyarrow",calculate_divisions=True) for item in output_phenotype_kmyo_df_path_list]

# trench_to_lookup_df_list = []
variant_to_trench_df_list = []

for i in range(len(output_phenotype_kmyo_df_list)):
    trench_to_lookup_df = output_phenotype_kmyo_df_list[i].set_index("phenotype trenchid",sorted=True).groupby('phenotype trenchid').apply(lambda x: x.iloc[0])
    trench_to_lookup_df = trench_to_lookup_df[['File Index','File Trench Index','lane orientation','oDEPool7_id']].compute()
    trench_to_lookup_df["oDEPool7_id Temp"] = copy.deepcopy(trench_to_lookup_df["oDEPool7_id"])
    trench_to_lookup_df["Experiment #"] = i
    variant_to_trench_df = trench_to_lookup_df.reset_index().set_index("oDEPool7_id")
    variant_to_trench_df_list.append(variant_to_trench_df)
    
variant_to_trench_df = pd.concat(variant_to_trench_df_list,axis=0).sort_index()
variant_to_trench_dd = dd.from_pandas(variant_to_trench_df,npartitions=200).persist()

In [ ]:
headpath_list = ["/home/de64/scratch/de64/sync_folder/2022-06-04_lDE23_Run_1/Growth_Division",\
                 "/home/de64/scratch/de64/sync_folder/2022-06-11_lDE23_Run_2_TimeSubsample/Growth_Division"]
output_path = '/home/de64/scratch/de64/sync_folder/2022-08-10_lDE23_LowSampling_Merged_Analysis/AVI_Library/'
channel_params_dict = {"mCherry":(0,10000,10)}

variant_to_trench_dd.groupby("oDEPool7_id").apply(lambda x: lookup_to_avi(headpath_list,output_path,channel_params_dict,x)).compute()

In [ ]:
#check that number of images agrees with merged df when export done

In [ ]:
dask_controller.shutdown()

### Interactive Variant Overlay

In [ ]:
output_phenotype_kmyo_df = dd.read_parquet("/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/2022-07-01_lDE23_Final_Barcodes_df/",calculate_divisions=True)

In [ ]:
output_phenotype_kmyo_df.index

In [ ]:
headpath = "/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/Growth_Division/"
#note: shutdown dask when doing this...fix bug later
overlay_handle = tr.variant_overlay(headpath,"/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/2022-07-01_lDE23_Final_Barcodes_df/",
                                 display_values_list=["Gene","TargetID","N Mismatch"],persist_data=False)  ##fix this, was improperly made (only initial cellID timepoints)

In [ ]:
overlay_handle.view_overlay(vmin=0,vmax=10000)

### Interactive Variant Table

In [ ]:
gene_table = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/",engine='pyarrow',calculate_divisions=True)
gene_table = gene_table.reset_index().set_index("phenotype trenchid", sorted=True)
gene_table = gene_table.groupby("phenotype trenchid",sort=False).apply(lambda x: x.iloc[0]).reset_index().set_index("FOV Parquet Index")
gene_table_out = gene_table.groupby("sgRNA").apply(lambda x: x.iloc[0])
gene_table_out["phenotype trenchids"] = gene_table.groupby("sgRNA").apply(lambda x: x["phenotype trenchid"].tolist())
gene_table_out = gene_table_out[["Gene",'Target Sequence',"phenotype trenchids","N Mismatch","N Target Sites","Category","Strand","oDEPool7_id"]].compute()

In [ ]:
kymo_xarr = tr.kymo_xarr("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/")
wrapped_kymo_xarr = tr.kymo_xarr("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division",unwrap=False)

In [ ]:
gene_table_layout,select_gene,select_trenchid,select_unpacked_trenchid = tr.linked_gene_table(gene_table_out,index_key='Gene',trenchids_as_list=True,trenchid_column='phenotype trenchids')

In [ ]:
gene_table_layout

In [ ]:
output_display,save_button = tr.linked_kymograph_for_gene_table(kymo_xarr,wrapped_kymo_xarr,gene_table_out,select_gene,select_trenchid,select_unpacked_trenchid=select_unpacked_trenchid,\
                                                    trenchid_column='phenotype trenchids',y_scale=1.,x_window_size=1400)

In [ ]:
output_display

In [ ]:
save_button

In [ ]:
sub_kymo_meta.loc[2721]

In [ ]:
trenchid_list = gene_table_out[gene_table_out["oDEPool7_id"]==22839]["phenotype trenchids"].tolist()[0]

In [ ]:
t_min = 100
t_max = 200
file_indices = []
trench_indices = []
for ind in trenchid_list:
    file_idx, trench_idx = sub_kymo_meta.loc[ind]
    file_indices.append(file_idx)
    trench_indices.append(trench_idx)
# make_fork_plot(nan_thr = 6, percentile = 50, t_min = t_min, t_max = t_max, trench_indices = trench_indices, file_idx = file_indices)
make_consensus_fork_plot(nan_thr = 6, percentile = 50, t_min = t_min, t_max = t_max, trench_indices = trench_indices, file_indices = file_indices)

In [ ]:
trench_indices = [0]
file_idx, trench_indices[0] = sub_kymo_meta.loc[10964]


In [ ]:
t_min = 100
t_max = 200
# make_fork_plot(nan_thr = 6, percentile = 50, t_min = 0, t_max = 200, trench_indices = [48], file_idx = 100)
make_fork_plot(nan_thr = 6, percentile = 50, t_min = t_min, t_max = t_max, trench_indices = trench_indices, file_idx = file_idx)

In [ ]:
import dask.dataframe as dd
kymo_meta = dd.read_parquet('/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata', engine='pyarrow', calculate_divisions=True)
sub_kymo_meta = kymo_meta[['File Index', 'File Trench Index', 'trenchid']].compute().set_index(['trenchid']).groupby('trenchid').apply(lambda x: x.iloc[0])



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import skimage as sk
%matplotlib inline

nan_thr = 6 #3
percentile = 50
t_min = 0
t_max = 200


file_idx = 100
trench_indices = [48]

def make_fork_plot(nan_thr = 6, percentile = 50, t_min = 0, t_max = 200, trench_indices = [48], file_idx = 100):

    with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
            gfp_data = infile["GFP-Penta"][:]

    with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
            seg_data = infile["data"][:]

    percentile_traces = []
    # trench_indices = range(gfp_data.shape[0])

    for idx in trench_indices:
        for t in range(t_min,t_max):
        # plt.imshow(seg_data[idx][t])
        # plt.show()
        # plt.imshow(gfp_data[idx][t])
        # plt.show()

            rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

            degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
            centroids = [rp.centroid for rp in rps]

            # Do it for the first one
            # i = 0
            for i,degree in enumerate(degrees):
                degree = degrees[0]
                centroid = centroids[0]
                masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
                # plt.imshow(masked_img)
                # plt.title('masked_img')
                # plt.show()

                rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
                # plt.imshow(rotated_img)
                # plt.title('Rotated img')
                # plt.show()
                rotated_img[rotated_img==0.]=np.NaN

                n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
                # plt.plot(n_not_nan, 'o')
                # plt.show()
                nan_mask = n_not_nan>nan_thr
                nan_masked_rotated_img = rotated_img[nan_mask]

                percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
                sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
                # percentile_traces.append(percentile_trace)
                # plt.plot(percentile_trace)
                # plt.title('median')
                # plt.show()
                # plt.plot(sum_trace)
                # plt.title('sum')
                percentile_traces.append(percentile_trace)

    percentile_traces = np.array(percentile_traces)
    trace_lens = np.array([len(trace) for trace in percentile_traces])
    trace_bins = np.sort(np.unique(trace_lens))
    med_traces = []
    for trace_len in trace_bins:
        trace_group = np.stack(percentile_traces[trace_lens==trace_len])
        med_trace = np.nanmedian(trace_group,axis=0)
        med_traces.append(med_trace)
    max_trace_len = np.max(trace_bins)
    trace_dif = max_trace_len-trace_bins
    padded_traces = np.array([np.pad(trace,((trace_dif[i]//2)+trace_dif[i]%2,trace_dif[i]//2),constant_values=np.NaN) for i,trace in enumerate(med_traces)])

    plt.imshow(padded_traces[:],vmin=300,vmax=500)

In [ ]:
def make_consensus_fork_plot(nan_thr = 6, percentile = 50, t_min = 0, t_max = 200, trench_indices = [48], file_indices = [100]):
    percentile_traces = []
    
    for i in range(len(trench_indices)):
        idx = trench_indices[i]
        with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_indices[i]) + ".hdf5", "r") as infile:
            gfp_data = infile["GFP-Penta"][:]

        with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_indices[i]) + ".hdf5", "r") as infile:
            seg_data = infile["data"][:]
        
        # trench_indices = range(gfp_data.shape[0])

        for t in range(t_min,t_max):
        # plt.imshow(seg_data[idx][t])
        # plt.show()
        # plt.imshow(gfp_data[idx][t])
        # plt.show()

            rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

            degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
            centroids = [rp.centroid for rp in rps]

            # Do it for the first one
            # i = 0
            for i,degree in enumerate(degrees):
                degree = degrees[0]
                centroid = centroids[0]
                masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
                # plt.imshow(masked_img)
                # plt.title('masked_img')
                # plt.show()

                rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
                # plt.imshow(rotated_img)
                # plt.title('Rotated img')
                # plt.show()
                rotated_img[rotated_img==0.]=np.NaN

                n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
                # plt.plot(n_not_nan, 'o')
                # plt.show()
                nan_mask = n_not_nan>nan_thr
                nan_masked_rotated_img = rotated_img[nan_mask]

                percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
                sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
                # percentile_traces.append(percentile_trace)
                # plt.plot(percentile_trace)
                # plt.title('median')
                # plt.show()
                # plt.plot(sum_trace)
                # plt.title('sum')
                percentile_traces.append(percentile_trace)

    percentile_traces = np.array(percentile_traces)
    trace_lens = np.array([len(trace) for trace in percentile_traces])
    trace_bins = np.sort(np.unique(trace_lens))
    med_traces = []
    for trace_len in trace_bins:
        trace_group = np.stack(percentile_traces[trace_lens==trace_len])
        med_trace = np.nanmedian(trace_group,axis=0)
        med_traces.append(med_trace)
    max_trace_len = np.max(trace_bins)
    trace_dif = max_trace_len-trace_bins
    padded_traces = np.array([np.pad(trace,((trace_dif[i]//2)+trace_dif[i]%2,trace_dif[i]//2),constant_values=np.NaN) for i,trace in enumerate(med_traces)])

    plt.imshow(padded_traces[:],vmin=300,vmax=500)